# Literature Survey - Automated / AI Data Extraction from Scientific Papers

*Context:* this survey was requested by my advisor to check whether others extract data from the
scientific literature in an automated / AI-driven way like our nAChR pipeline (`human_automation/`),
and to position our approach within that field.

**Bottom line:** Yes - it is an active field with two clear generations (rule/ML-based text-mining,
then LLM-based extraction). But the *specific* thing our pipeline does - automatically extracting the
**functional effect direction (LOF / GOF / no-net-effect)** from full-text electrophysiology using a
**local LLM** - sits in a genuine, under-served gap. Established variant-mining tools extract *which
variant is mentioned* and *variant<->disease* links, not *what the variant does functionally*; and the
functional ion-channel datasets are almost all curated **by hand**.

## At-a-glance: where each approach stands

| Tool / approach | What it pulls from papers | Method | Outputs functional LOF/GOF? |
|---|---|---|---|
| **tmVar / tmVar 3.0**, **LitVar 2.0** (NCBI) | variant *mentions*, normalized to dbSNP IDs | CRF / ML text-mining | No |
| **PubTator / PubTator3** (NCBI) | genes, diseases, variants (entity tags) | ML / transformer NER | No |
| **AVADA** (Stanford) | variants + genomic coordinates from **full text**, disease papers | 47 regexes + gradient-boosting ML | **No** (authors state it does not evaluate variants) |
| **Dagdelen et al. 2024** (*Nat. Commun.*) | arbitrary structured **JSON** records from papers | fine-tuned LLM (GPT-3 / Llama-2) | method is general (not variant-specific) |
| **BioMistral-7B**, local Llama/Mistral IE | clinical / biomedical structured fields | **local** open-source LLM | task-dependent |
| **ICEPO**, **FENICS** | electrophysiology parameters / annotations | **manual** curation + ontology | Yes - but manual |
| **funNCion**, **VPatho** | *predict* GOF vs LOF | ML trained on **curated** functional data | predicts (does not extract from text) |
| **OUR pipeline** | mutation + **LOF/GOF/no-effect** from full-text electrophysiology | **local LLM** (Qwen2.5-7B) + retrieval | **Yes - automated, directly from text** |

## Generation 1 - Classic biomedical text-mining for variants (rule / ML, ~2013-present)

A mature subfield exists *just* for pulling genetic variants out of papers:

- **tmVar / tmVar 3.0** (NCBI) - extracts sequence-variant *mentions* (HGVS notation) with conditional
  random fields, ~90% F-measure, and normalizes them to dbSNP IDs.
- **PubTator / PubTator3** (NCBI) - tags genes, diseases, and variants across all of PubMed/PMC.
- **LitVar 2.0** (NCBI, 2024) - built on tmVar + PubTator; tracks **~14 million unique variants
  (~70M mentions)** and links each to the literature it appears in.
- **AVADA** (Bejerano Lab, Stanford; *Genetics in Medicine* 2020) - the closest classic analog to us:
  it reads **full-text** Mendelian-disease papers (47 hand-built regexes + a gradient-boosting
  classifier) and retrieves variant evidence + genomic coordinates, recovering ~60% of HGMD variants
  (203k variants from 61k articles).

**Critical limitation of this whole generation:** they extract *which variant is mentioned where* and
*variant<->disease* links - **not what the variant does functionally**. AVADA's authors explicitly
state it does *not* attempt to evaluate variants. None of these output LOF/GOF.

## Generation 2 - LLMs for structured extraction from papers (our method's category, 2023-present)

This is the new wave, and it is exactly the paradigm our pipeline uses:

- **Dagdelen et al., *Nature Communications* 2024 - "Structured information extraction from scientific
  text with large language models"** is the canonical reference: fine-tune GPT-3 / Llama-2 to read a
  paper and emit **JSON records** of structured facts. Domain-general (demoed on materials science).
  This is the same paradigm as our pipeline: *paper text -> LLM -> validated JSON rows*.
- **Reviews / benchmarks** confirm the momentum: a 2024 review of *Scientific Knowledge Extraction
  using LLMs in Biomedical Sciences*, and a 2025 *Nature Communications* benchmark of LLMs for
  biomedical NLP.
- In **systematic-review automation**, LLMs reach **80-94% data-extraction accuracy** and cut screening
  workload ~40% - *but* consistently require human checking (which is why our rows are `Pending`).
- **Local / open-source models - exactly our setup:** a clear 2025 trend toward running open models
  *locally* for extraction (privacy, cost, no API limits): **BioMistral-7B**, local **Llama / Mistral**
  medication & clinical extraction (JAMIA Open 2025), the `llm_extractinator` framework. We run a local
  **Qwen2.5-7B** for the same reasons (and because Gemini's free tier capped us at 20 requests/day).

## Generation 3 - Our exact niche: ion channels & *functional* effects

- The functional ion-channel resources are **almost entirely manually curated**: **ICEPO** (ion-channel
  electrophysiology ontology), **FENICS** (1,484 hand-curated functional annotations in sodium
  channels), and prediction tools like **funNCion** and **VPatho** that *predict* GOF/LOF but train on
  **manually curated** functional datasets.
- ~10,000-15,000 ion-channel papers are published per year - manual functional curation cannot keep
  pace. That is the bottleneck.
- **No established tool automatically *extracts the LOF/GOF call* from full-text electrophysiology.**
  The variant-mining tools (Gen 1) skip function; the functional databases (Gen 3) are curated by hand.

## Where our work fits (the contribution)

We sit at the **intersection of two mature areas, in a real gap** - combining (a) multi-source
retrieval + free keyword ranking with (b) a **local LLM that extracts the *functional effect
direction* (LOF / GOF / no-net-effect)** from electrophysiology, for nAChRs specifically:

- Gen-1 tools do variant<->disease, **not** function -> we add function.
- Gen-3 functional datasets are **manual** -> we automate that curation step.
- And we do it **fully locally and free**, with **human-in-the-loop** review (the `Pending` flag) -
  which is the field's documented best practice.

**Honest framing (shows rigor):** we are *not* claiming a brand-new method - LLM->JSON extraction is
established (Dagdelen 2024). The contribution is **applying it to automate functional-effect curation
for a channel family where that is still done by hand**, and showing an open *local* model can do it
for free. Known limitations are the field's own: LLM accuracy (~80-94%, so review is mandatory) and
paywall-bounded recall.

## Sources

- [LitVar 2.0 - tracking variants in literature (NCBI/NAR)](https://pmc.ncbi.nlm.nih.gov/articles/PMC11096795/)
- [tmVar - text-mining sequence variants (Bioinformatics 2013)](https://academic.oup.com/bioinformatics/article/29/11/1433/220291)
- [tmVar 3.0 (arXiv)](https://arxiv.org/pdf/2204.03637)
- [AVADA - automated variant evidence from full text (PMC)](https://pmc.ncbi.nlm.nih.gov/articles/PMC7301356/) | [AVADA site, Stanford](http://bejerano.stanford.edu/AVADA/)
- [Dagdelen et al. - Structured information extraction with LLMs, *Nature Communications* 2024](https://www.nature.com/articles/s41467-024-45563-x)
- [Review: Scientific Knowledge Extraction using LLMs in Biomedical Sciences (arXiv 2024)](https://arxiv.org/html/2412.03531v1)
- [Benchmarking LLMs for biomedical NLP, *Nature Communications* 2025](https://www.nature.com/articles/s41467-025-56989-2)
- [LLMs in medical research: systematic reviews -> clinical studies (MDPI 2026)](https://www.mdpi.com/2306-5354/13/3/365)
- [Medication information extraction using *local* LLMs (ScienceDirect 2025)](https://www.sciencedirect.com/science/article/pii/S1532046425001273)
- [Open-source LLMs for clinical information extraction (JAMIA Open 2025)](https://academic.oup.com/jamiaopen/article/8/5/ooaf109/8270821)
- [ICEPO - ion channel electrophysiology ontology (Database 2016)](https://academic.oup.com/database/article/doi/10.1093/database/baw017/2630205)
- [Optimizing functional evidence in epilepsy ion-channel variants (bioRxiv 2024)](https://www.biorxiv.org/content/10.1101/2024.05.09.593343.full.pdf)
- [VPatho - predicting GOF/LOF variants (Briefings in Bioinformatics 2023)](https://academic.oup.com/bib/article/24/1/bbac535/6931725)

*Survey compiled June 2026. Generated as part of the nAChR `human_automation` data-extraction project.*